# CRDW v4 Analysis: Cardiac and Pharmacogenomic Vulnerability in Treatment-Resistant OUD

**Purpose:** Companion CRDW analysis for Paper 2 (Special Communication + Full Article). Validates All of Us findings with active-window CYP2D6 inhibitor exposure and operationalizes cumulative hERG blockade load.

**Key differences from prior CRDW runs:**
1. **Candidate/comparator funnel** — stratifies OUD by treatment-refractory qualifiers (PTSD, depression, anxiety, MOUD)
2. **Active-window phenoconversion** — 90-day lookback for CYP2D6 inhibitors (vs "ever prescribed")
3. **Cumulative hERG load** — potency-weighted sum across regimen (Tier 1 categorical)
4. **Compound risk cross-tabulation** — cardiac × CYP2D6 × hERG (2×2×2)
5. **Sensitivity analyses** — methadone-excluded, qualifier gradient, age 25–55

**Output Tables:**
- Table 1: Candidate funnel
- Table 2: QT/TdP risk markers (candidates vs comparator)
- Table 3: CYP2D6 inhibitor exposure (active-window)
- Table 4: hERG cumulative load distribution
- Table 5: Compound risk cross-tabulation
- Table 6: Screening burden scenarios

**Author:** James Green  
**Date:** April 2026  
**Platform:** AMAREL (Rutgers HPC)

---

In [1]:
# =============================================================================
# CELL 1: CONFIGURATION & IMPORTS (v4.1 - corrected medication filtering)
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import warnings
import os
import re
warnings.filterwarnings('ignore')

DATA_DIR = '/scratch/bl696/Hypertension/'
MEDICATIONS_FILE = DATA_DIR + 'Hypertension Medications 20250417.csv'
COMORBIDITIES_FILE = DATA_DIR + 'Hypertension With Comorbidities 20250404.csv'

# New output directory so v4 outputs remain available for comparison
OUTPUT_DIR = os.path.expanduser('~/paper2_crdw_v4_1/')
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_ENCODING = 'cp1252'
CHUNK_SIZE = 50000

DX_ID_COL = 'PAT_MRN_ID_ENCRYPT'
DX_CODE_COL = 'CURRENT_ICD10_LIST'
AGE_COL = 'PATIENT_AGE'
SEX_COL = 'SEX'
RACE_COL = 'RACE'
MED_ID_COL = 'PAT_MRN_ID_ENCRYPT'

# KEY CHANGE from v4: use SIMPLE_GENERIC as primary; fall back to MEDICATION_NAME
MED_NAME_COL = 'SIMPLE_GENERIC'
MED_NAME_FALLBACK_COL = 'MEDICATION_NAME'

# Columns needed for the v4.1 filters
MED_STATUS_COL = 'ACTIVE_ORDER_STATUS'
MED_ROUTE_COL = 'MED_ADMIN_ROUTE'

MED_DATE_COL = 'ORDER_DATE'
DX_DATE_COL = 'DIAGNOSIS_DATE'

# ============================================================================
# v4.1 FILTER TOGGLES
# ============================================================================
EXCLUDE_DISCONTINUED = True
DISCONTINUED_STATUS_VALUES = {'Discontinued Medication', 'Discontinued'}

EXCLUDE_IV = True
IV_ROUTE_SUBSTRINGS = ['intravenous', 'injection', 'iv push', 'iv infusion',
                       'intraosseous', 'intra-arterial']
IV_NAME_PATTERNS = [
    r'\\bintravenous\\b',
    r'\\bin 0\\.9\\s*%?\\s*sodium',
    r'\\bin nacl\\b',
    r'mg/ml\\s*intravenous',
    r'\\bin dextrose\\b',
    r'\\biso-osm\\b',
    r'\\biv drip\\b',
    r'\\bfor injection\\b',
    r'\\binjection\\b',
]
IV_NAME_PATTERN_RE = re.compile('|'.join(IV_NAME_PATTERNS), re.IGNORECASE)

ACTIVE_WINDOW_DAYS = 90
STUDY_START = '2018-01-01'
STUDY_END = '2025-12-31'

print("CRDW v4.1 Analysis - Paper 2 (CORRECTED MEDICATION FILTERS)")
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Output dir: {OUTPUT_DIR}")
print("Filter changes vs v4:")
print(f"  - Primary drug name column: {MED_NAME_COL} (fallback: {MED_NAME_FALLBACK_COL})")
print(f"  - Exclude discontinued orders: {EXCLUDE_DISCONTINUED}")
print(f"  - Exclude IV formulations:     {EXCLUDE_IV}")
print("=" * 70)


CRDW v4.1 Analysis - Paper 2 (CORRECTED MEDICATION FILTERS)
Run date: 2026-04-21 23:20
Output dir: /home/jg1984/paper2_crdw_v4_1/
Filter changes vs v4:
  - Primary drug name column: SIMPLE_GENERIC (fallback: MEDICATION_NAME)
  - Exclude discontinued orders: True
  - Exclude IV formulations:     True


In [2]:
# =============================================================================
# CELL 2: VERIFY FILE SCHEMA (v4.1 - also verifies status & route columns)
# =============================================================================
print("Checking comorbidities file...")
dx_sample = pd.read_csv(COMORBIDITIES_FILE, encoding=FILE_ENCODING, nrows=5)
print(f"  Columns ({len(dx_sample.columns)})")

print("\nChecking medications file...")
med_sample = pd.read_csv(MEDICATIONS_FILE, encoding=FILE_ENCODING, nrows=5)
print(f"  Columns ({len(med_sample.columns)}): {list(med_sample.columns)}")

for col in [DX_ID_COL, DX_CODE_COL, AGE_COL, SEX_COL]:
    if col not in dx_sample.columns:
        print(f"  WARNING: Missing in comorbidities: {col}")

required_med_cols = [MED_ID_COL, MED_NAME_COL, MED_NAME_FALLBACK_COL,
                     MED_STATUS_COL, MED_ROUTE_COL]
print("\nRequired v4.1 medications columns:")
for col in required_med_cols:
    present = col in med_sample.columns
    tag = "OK " if present else "MISSING"
    print(f"  [{tag}] {col}")

# Preview distinct status/route values so the analyst can confirm the
# filter targets actually occur in this extract
try:
    prev = pd.read_csv(MEDICATIONS_FILE, encoding=FILE_ENCODING,
                       usecols=[MED_STATUS_COL, MED_ROUTE_COL], nrows=20000)
    print(f"\nTop {MED_STATUS_COL} values (first 20k rows):")
    for v, c in prev[MED_STATUS_COL].value_counts().head(10).items():
        flag = "  <-- WILL BE EXCLUDED" if str(v) in DISCONTINUED_STATUS_VALUES else ""
        print(f"  {str(v)[:40]:40s} {c:8,}{flag}")
    print(f"\nTop {MED_ROUTE_COL} values (first 20k rows):")
    for v, c in prev[MED_ROUTE_COL].value_counts().head(10).items():
        flag = ""
        if pd.notna(v):
            vl = str(v).lower()
            if any(sub in vl for sub in IV_ROUTE_SUBSTRINGS):
                flag = "  <-- WILL BE EXCLUDED"
        print(f"  {str(v)[:40]:40s} {c:8,}{flag}")
except Exception as e:
    print(f"Status/route preview failed: {e}")

HAS_MED_DATE = MED_DATE_COL in med_sample.columns
HAS_DX_DATE = DX_DATE_COL in dx_sample.columns
print(f"\nHAS_MED_DATE = {HAS_MED_DATE}")
print(f"HAS_DX_DATE  = {HAS_DX_DATE}")
print("=" * 70)


Checking comorbidities file...
  Columns (14)

Checking medications file...
  Columns (56): ['PAT_MRN_ID_ENCRYPT', 'PAT_ENC_CSN_ID_ENCRYPT', 'ORDER_MED_ID_ENCRYPT', 'ORD_DAYS_SINCE_DX_FRST_OBSRVD', 'DISPENSABLE_MED_ID', 'MEDICATION_NAME', 'NDC_CODE_LIST_1', 'NDC_CODE_LIST_2', 'MED_FORM', 'MED_STRENGTH', 'MED_ADMIN_ROUTE', 'THERAPY_CLASS', 'PHARMACUETICAL_CLASS', 'PHARMACEUTICAL_SUBCLASS', 'SIMPLE_GENERIC', 'DEA_CLASS', 'CONCENTRATION_NAME', 'ORDER_CLASS', 'ORDER_STATUS', 'ACTIVE_ORDER_STATUS', 'IS_PENDING_ORD_YN', 'PENDING_APPROVAL_FLAG', 'DONOT_DISP_YN', 'ORD_COMP_YN', 'STRT_DAYS_SINCE_DX_FRST_OBSRVD', 'END_DAYS_SINCE_DX_FRST_OBSRVD', 'REASON_FOR_DISCON', 'DSCNT_DAYS_SINCE_DX_FRST_OBSRVD', 'PAT_SUPP_MED_YN', 'HV_IS_SELF_ADM_YN', 'NUMBER_OF_DOSES', 'DOSES_REMAINING', 'PTA_LAST_DOSE_TIME', 'IS_FAM_YN', 'FAM_LAST_ADMIN_INST', 'HV_DISCRETE_DOSE', 'HV_DOSE_UNIT', 'FREQ_NAME', 'MIN_DISCRETE_DOSE', 'MAX_DISCRETE_DOSE', 'DOSE_UNIT', 'MED_DIS_DISP_QTY', 'MED_DIS_DISP_UNIT', 'MIN_RATE', 'MAX_RA

In [3]:
# =============================================================================
# CELL 3: CODE SETS AND DRUG LISTS (v4)
# =============================================================================

# ----- OUD / Candidate Qualifier Codes -----
OUD_PREFIXES = ['F11']  # Opioid-related disorders

# Candidate funnel qualifiers (ICD-10)
PTSD_PREFIXES = ['F43.1']  # PTSD
DEPRESSION_PREFIXES = ['F32', 'F33']  # MDD single episode + recurrent
ANXIETY_PREFIXES = ['F41']  # Anxiety disorders
TBI_PREFIXES = ['S06']  # Intracranial injury
TBI_HISTORY_CODES = ['Z87.820']  # Personal history of TBI

# ----- Cardiac/Renal Flags (AHA/ACC framework) -----
LONG_QT_CODES = ['I45.81']
VARR_CODES = ['I47.2']  # Ventricular tachycardia
VARR_PREFIXES = ['I49']  # Other cardiac arrhythmias
HF_PREFIXES = ['I50']  # Heart failure
CKD_3_4_CODES = ['N18.30', 'N18.31', 'N18.32', 'N18.4']  # CKD 3a, 3b, 3 unspec, 4
CKD_3_4_PREFIXES = ['N18.3']  # Catch variants
CKD_5_ESRD_CODES = ['N18.5', 'N18.6']  # CKD 5, ESRD

# ----- CYP2D6 Inhibitors (Phenoconversion) -----
# Categorized by inhibition strength per FDA/CPIC guidance
CYP2D6_STRONG = ['bupropion', 'paroxetine', 'paxil', 'wellbutrin',
                 'fluoxetine', 'prozac', 'terbinafine', 'quinidine']
CYP2D6_MODERATE = ['duloxetine', 'cymbalta', 'mirabegron', 'cinacalcet', 'abiraterone']
CYP2D6_WEAK = ['sertraline', 'zoloft', 'diphenhydramine', 'benadryl',
               'amiodarone', 'hydroxychloroquine', 'cimetidine', 'celecoxib']

# ----- hERG Potency Tiers (per published IC50) -----
# Tier 1 categorical weights: HIGH=3, MODERATE=1, LOW=0.3
HERG_HIGH = {  # IC50 < 1 µM
    'haloperidol': 3, 'haldol': 3,
    'droperidol': 3,
    'domperidone': 3,
    'thioridazine': 3,
    'pimozide': 3,
    'ondansetron': 3, 'zofran': 3,
    'dolasetron': 3,
    'granisetron': 3,
    'risperidone': 3, 'risperdal': 3,
    'ziprasidone': 3, 'geodon': 3,
    'paroxetine': 3, 'paxil': 3,  # Note: also strong CYP2D6 inhibitor
    'aripiprazole': 3, 'abilify': 3,
}

HERG_MODERATE = {  # IC50 1-10 µM
    'methadone': 1,
    'amiodarone': 1,
    'sotalol': 1,
    'escitalopram': 1, 'lexapro': 1,
    'citalopram': 1, 'celexa': 1,  # Careful: escitalopram contains citalopram substring
    'fluoxetine': 1, 'prozac': 1,
    'sertraline': 1, 'zoloft': 1,
    'duloxetine': 1, 'cymbalta': 1,
    'hydroxyzine': 1, 'vistaril': 1, 'atarax': 1,
    'trazodone': 1, 'desyrel': 1,
    'quetiapine': 1, 'seroquel': 1,
    'olanzapine': 1, 'zyprexa': 1,
    'amitriptyline': 1, 'elavil': 1,
    'nortriptyline': 1, 'pamelor': 1,
    'doxepin': 1,
    'imipramine': 1,
    'desipramine': 1,
}

HERG_LOW = {  # IC50 > 10 µM
    'diphenhydramine': 0.3, 'benadryl': 0.3,
    'loratadine': 0.3,
    'azithromycin': 0.3,
    'erythromycin': 0.3,
}

# ----- MOUD agents -----
MOUD_METHADONE = ['methadone']
MOUD_BUPRENORPHINE = ['buprenorphine', 'suboxone', 'subutex', 'sublocade', 'zubsolv']
MOUD_NALTREXONE = ['naltrexone', 'vivitrol', 'revia']

print("Code sets loaded:")
print(f"  OUD prefixes: {OUD_PREFIXES}")
print(f"  Candidate qualifiers: PTSD, Depression, Anxiety, TBI, ≥2 MOUD agents")
print(f"  CYP2D6 inhibitors: {len(CYP2D6_STRONG)} strong, {len(CYP2D6_MODERATE)} moderate, {len(CYP2D6_WEAK)} weak")
print(f"  hERG drugs: {len(HERG_HIGH)} high-potency, {len(HERG_MODERATE)} moderate, {len(HERG_LOW)} low")
print("="*70)

Code sets loaded:
  OUD prefixes: ['F11']
  Candidate qualifiers: PTSD, Depression, Anxiety, TBI, ≥2 MOUD agents
  CYP2D6 inhibitors: 8 strong, 5 moderate, 8 weak
  hERG drugs: 18 high-potency, 29 moderate, 5 low


In [4]:
# =============================================================================
# CELL 4: BUILD OUD COHORT + INDEX DATE
# =============================================================================
# Identify all F11.x patients. Index date = most recent OUD encounter (if date
# available) or placeholder for "ever prescribed" fallback.
# =============================================================================
print("[1/N] Building OUD cohort...\n")

patient_data = {}  # {mrn: {'age':, 'sex':, 'index_date':, 'diagnoses':set(), ...}}
chunk_count = 0

for chunk in pd.read_csv(COMORBIDITIES_FILE, encoding=FILE_ENCODING,
                         chunksize=CHUNK_SIZE, low_memory=False):
    chunk_count += 1
    codes = chunk[DX_CODE_COL].astype(str)
    mask = codes.str.startswith('F11', na=False)
    oud_chunk = chunk[mask]

    for _, row in oud_chunk.iterrows():
        mrn = row[DX_ID_COL]
        if mrn not in patient_data:
            patient_data[mrn] = {
                'age': row.get(AGE_COL, None),
                'sex': row.get(SEX_COL, None),
                'race': row.get(RACE_COL, None) if RACE_COL in chunk.columns else None,
                'diagnoses': set(),
                'index_date': None,
            }
        # Update index_date if we have dx_date
        if HAS_DX_DATE:
            try:
                dx_date = pd.to_datetime(row[DX_DATE_COL], errors='coerce')
                if pd.notna(dx_date):
                    if patient_data[mrn]['index_date'] is None or dx_date > patient_data[mrn]['index_date']:
                        patient_data[mrn]['index_date'] = dx_date
            except Exception:
                pass
    if chunk_count % 20 == 0:
        print(f"  Processed {chunk_count} chunks, {len(patient_data):,} OUD patients so far...")

oud_patients = set(patient_data.keys())
print(f"\n✓ Total OUD patients: {len(oud_patients):,}")

if HAS_DX_DATE:
    with_dates = sum(1 for p in patient_data.values() if p['index_date'] is not None)
    print(f"  Patients with index date: {with_dates:,} ({100*with_dates/len(oud_patients):.1f}%)")
else:
    print("  ⚠️  No diagnosis dates — index_date set to None, temporal windowing disabled")

print("="*70)

[1/N] Building OUD cohort...

  Processed 20 chunks, 123 OUD patients so far...
  Processed 40 chunks, 244 OUD patients so far...
  Processed 60 chunks, 387 OUD patients so far...
  Processed 80 chunks, 512 OUD patients so far...
  Processed 100 chunks, 607 OUD patients so far...
  Processed 120 chunks, 745 OUD patients so far...
  Processed 140 chunks, 865 OUD patients so far...
  Processed 160 chunks, 975 OUD patients so far...
  Processed 180 chunks, 1,088 OUD patients so far...
  Processed 200 chunks, 1,196 OUD patients so far...
  Processed 220 chunks, 1,307 OUD patients so far...
  Processed 240 chunks, 1,438 OUD patients so far...
  Processed 260 chunks, 1,574 OUD patients so far...
  Processed 280 chunks, 1,684 OUD patients so far...
  Processed 300 chunks, 1,729 OUD patients so far...
  Processed 320 chunks, 1,859 OUD patients so far...
  Processed 340 chunks, 2,155 OUD patients so far...
  Processed 360 chunks, 2,420 OUD patients so far...
  Processed 380 chunks, 2,697 OUD pa

In [5]:
# =============================================================================
# CELL 5: COLLECT ALL DIAGNOSES FOR OUD PATIENTS
# =============================================================================
print("[2/N] Collecting all diagnoses for OUD cohort...\n")

chunk_count = 0
for chunk in pd.read_csv(COMORBIDITIES_FILE, encoding=FILE_ENCODING,
                         chunksize=CHUNK_SIZE, low_memory=False):
    chunk_count += 1
    chunk_oud = chunk[chunk[DX_ID_COL].isin(oud_patients)]
    for _, row in chunk_oud.iterrows():
        mrn = row[DX_ID_COL]
        if mrn in patient_data:
            patient_data[mrn]['diagnoses'].add(str(row[DX_CODE_COL]))
    if chunk_count % 20 == 0:
        print(f"  Processed {chunk_count} chunks...")

total_dx = sum(len(p['diagnoses']) for p in patient_data.values())
print(f"\n✓ Total diagnosis codes collected: {total_dx:,}")
print(f"  Mean per patient: {total_dx/len(patient_data):.1f}")
print("="*70)

[2/N] Collecting all diagnoses for OUD cohort...

  Processed 20 chunks...
  Processed 40 chunks...
  Processed 60 chunks...
  Processed 80 chunks...
  Processed 100 chunks...
  Processed 120 chunks...
  Processed 140 chunks...
  Processed 160 chunks...
  Processed 180 chunks...
  Processed 200 chunks...
  Processed 220 chunks...
  Processed 240 chunks...
  Processed 260 chunks...
  Processed 280 chunks...
  Processed 300 chunks...
  Processed 320 chunks...
  Processed 340 chunks...
  Processed 360 chunks...
  Processed 380 chunks...
  Processed 400 chunks...
  Processed 420 chunks...
  Processed 440 chunks...
  Processed 460 chunks...
  Processed 480 chunks...
  Processed 500 chunks...
  Processed 520 chunks...
  Processed 540 chunks...
  Processed 560 chunks...
  Processed 580 chunks...
  Processed 600 chunks...
  Processed 620 chunks...

✓ Total diagnosis codes collected: 197,069
  Mean per patient: 44.2


In [6]:
# =============================================================================
# CELL 6: CANDIDATE FUNNEL — ASSIGN QUALIFIER FLAGS
# =============================================================================
# Qualifiers: PTSD, Depression, Anxiety, TBI, ≥2 MOUD agents
# (MOUD flag populated in next cell after medication extraction)
# =============================================================================

def has_prefix(diagnoses, prefixes):
    return any(any(d.startswith(p) for p in prefixes) for d in diagnoses)

def has_code(diagnoses, codes):
    return any(d in codes for d in diagnoses)

print("[3/N] Assigning candidate funnel flags...\n")

for mrn, pdata in patient_data.items():
    dx = pdata['diagnoses']
    pdata['has_ptsd'] = has_prefix(dx, PTSD_PREFIXES)
    pdata['has_depression'] = has_prefix(dx, DEPRESSION_PREFIXES)
    pdata['has_anxiety'] = has_prefix(dx, ANXIETY_PREFIXES)
    pdata['has_tbi'] = has_prefix(dx, TBI_PREFIXES) or has_code(dx, TBI_HISTORY_CODES)
    # Cardiac/renal flags
    pdata['has_long_qt'] = has_code(dx, LONG_QT_CODES)
    pdata['has_varr'] = has_code(dx, VARR_CODES) or has_prefix(dx, VARR_PREFIXES)
    pdata['has_hf'] = has_prefix(dx, HF_PREFIXES)
    pdata['has_ckd_3_4'] = has_code(dx, CKD_3_4_CODES) or has_prefix(dx, CKD_3_4_PREFIXES)
    pdata['has_ckd_5_esrd'] = has_code(dx, CKD_5_ESRD_CODES)
    pdata['has_any_cardiac_renal'] = any([
        pdata['has_long_qt'], pdata['has_varr'], pdata['has_hf'],
        pdata['has_ckd_3_4'], pdata['has_ckd_5_esrd']
    ])

# Summary counts
def count_flag(flag):
    return sum(1 for p in patient_data.values() if p.get(flag, False))

n = len(patient_data)
print("Qualifier prevalence in OUD cohort:")
print(f"  PTSD:       {count_flag('has_ptsd'):,} ({100*count_flag('has_ptsd')/n:.1f}%)")
print(f"  Depression: {count_flag('has_depression'):,} ({100*count_flag('has_depression')/n:.1f}%)")
print(f"  Anxiety:    {count_flag('has_anxiety'):,} ({100*count_flag('has_anxiety')/n:.1f}%)")
print(f"  TBI:        {count_flag('has_tbi'):,} ({100*count_flag('has_tbi')/n:.1f}%)")
print()
print("Cardiac/renal prevalence:")
print(f"  Long QT:    {count_flag('has_long_qt'):,} ({100*count_flag('has_long_qt')/n:.1f}%)")
print(f"  V-tach:     {count_flag('has_varr'):,} ({100*count_flag('has_varr')/n:.1f}%)")
print(f"  HF:         {count_flag('has_hf'):,} ({100*count_flag('has_hf')/n:.1f}%)")
print(f"  CKD 3-4:    {count_flag('has_ckd_3_4'):,} ({100*count_flag('has_ckd_3_4')/n:.1f}%)")
print(f"  CKD 5/ESRD: {count_flag('has_ckd_5_esrd'):,} ({100*count_flag('has_ckd_5_esrd')/n:.1f}%)")
print(f"  Any:        {count_flag('has_any_cardiac_renal'):,} ({100*count_flag('has_any_cardiac_renal')/n:.1f}%)")
print("="*70)

[3/N] Assigning candidate funnel flags...

Qualifier prevalence in OUD cohort:
  PTSD:       356 (8.0%)
  Depression: 1,598 (35.8%)
  Anxiety:    1,845 (41.4%)
  TBI:        93 (2.1%)

Cardiac/renal prevalence:
  Long QT:    34 (0.8%)
  V-tach:     229 (5.1%)
  HF:         669 (15.0%)
  CKD 3-4:    339 (7.6%)
  CKD 5/ESRD: 138 (3.1%)
  Any:        1,065 (23.9%)


In [7]:
# =============================================================================
# CELL 7: EXTRACT MEDICATIONS WITH v4.1 CORRECTED FILTERS
# =============================================================================
# Three filters applied during the chunked read:
#   1. Drug name from SIMPLE_GENERIC (fallback to MEDICATION_NAME if empty)
#   2. Reject records where ACTIVE_ORDER_STATUS in DISCONTINUED_STATUS_VALUES
#   3. Reject IV formulations via MED_ADMIN_ROUTE substring match OR
#      MEDICATION_NAME pattern match (raw name preserves formulation text)
#
# Rejection counters quantify the magnitude of correction vs v4.
# =============================================================================
print("[4/N] Extracting medications with v4.1 corrected filters...\n")

for mrn in patient_data:
    patient_data[mrn]['meds'] = []

chunk_count = 0
total_oud_records = 0
kept = 0
rej_empty_name = 0
rej_discontinued = 0
rej_iv_route = 0
rej_iv_name = 0

def is_iv_record(route_val, name_val):
    if pd.notna(route_val):
        rv = str(route_val).lower()
        for sub in IV_ROUTE_SUBSTRINGS:
            if sub in rv:
                return True, 'route'
    if pd.notna(name_val):
        if IV_NAME_PATTERN_RE.search(str(name_val)):
            return True, 'name'
    return False, None

for chunk in pd.read_csv(MEDICATIONS_FILE, encoding=FILE_ENCODING,
                         chunksize=CHUNK_SIZE, low_memory=False):
    chunk_count += 1
    chunk_oud = chunk[chunk[MED_ID_COL].isin(oud_patients)]
    total_oud_records += len(chunk_oud)

    for _, row in chunk_oud.iterrows():
        mrn = row[MED_ID_COL]
        if mrn not in patient_data:
            continue

        # Filter 1: SIMPLE_GENERIC preferred, MEDICATION_NAME fallback
        generic = row.get(MED_NAME_COL)
        fallback = row.get(MED_NAME_FALLBACK_COL)
        if pd.notna(generic) and str(generic).strip():
            med_name = str(generic).lower().strip()
        elif pd.notna(fallback) and str(fallback).strip():
            med_name = str(fallback).lower().strip()
        else:
            rej_empty_name += 1
            continue

        # Filter 2: discontinued orders
        if EXCLUDE_DISCONTINUED:
            status = row.get(MED_STATUS_COL)
            if pd.notna(status) and str(status).strip() in DISCONTINUED_STATUS_VALUES:
                rej_discontinued += 1
                continue

        # Filter 3: IV formulations - check route AND raw name
        if EXCLUDE_IV:
            is_iv, reason = is_iv_record(
                row.get(MED_ROUTE_COL),
                row.get(MED_NAME_FALLBACK_COL)
            )
            if is_iv:
                if reason == 'route':
                    rej_iv_route += 1
                else:
                    rej_iv_name += 1
                continue

        order_date = None
        if HAS_MED_DATE:
            try:
                order_date = pd.to_datetime(row[MED_DATE_COL], errors='coerce')
                if pd.isna(order_date):
                    order_date = None
            except Exception:
                order_date = None

        patient_data[mrn]['meds'].append((med_name, order_date))
        kept += 1

    if chunk_count % 20 == 0:
        print(f"  Processed {chunk_count} chunks, kept={kept:,} / scanned={total_oud_records:,}")

print(f"\nExtraction complete (v4.1 filtered):")
print(f"  OUD records scanned:       {total_oud_records:,}")
print(f"  Records kept:              {kept:,}")
print(f"  Rejections:")
print(f"    empty drug name:         {rej_empty_name:,}")
print(f"    discontinued status:     {rej_discontinued:,}")
print(f"    IV formulation (route):  {rej_iv_route:,}")
print(f"    IV formulation (name):   {rej_iv_name:,}")
total_rej = rej_empty_name + rej_discontinued + rej_iv_route + rej_iv_name
pct = 100.0 * total_rej / max(total_oud_records, 1)
print(f"  Total rejection rate:      {pct:.1f}% of OUD med records")
print(f"  Mean kept per patient:     {kept/max(len(patient_data),1):.1f}")
print("=" * 70)


[4/N] Extracting medications with v4.1 corrected filters...

  Processed 20 chunks, kept=13,858 / scanned=29,395
  Processed 40 chunks, kept=26,896 / scanned=55,863
  Processed 60 chunks, kept=37,629 / scanned=81,100
  Processed 80 chunks, kept=48,376 / scanned=105,062
  Processed 100 chunks, kept=58,402 / scanned=127,577
  Processed 120 chunks, kept=67,960 / scanned=149,348
  Processed 140 chunks, kept=77,770 / scanned=171,743
  Processed 160 chunks, kept=86,893 / scanned=192,889
  Processed 180 chunks, kept=95,507 / scanned=214,708
  Processed 200 chunks, kept=106,189 / scanned=238,800
  Processed 220 chunks, kept=117,053 / scanned=264,314
  Processed 240 chunks, kept=125,271 / scanned=287,140
  Processed 260 chunks, kept=133,991 / scanned=311,865
  Processed 280 chunks, kept=145,214 / scanned=340,330
  Processed 300 chunks, kept=155,100 / scanned=365,259
  Processed 320 chunks, kept=165,311 / scanned=390,118
  Processed 340 chunks, kept=176,763 / scanned=414,120
  Processed 360 chun

In [8]:
# =============================================================================
# CELL 8: MOUD FLAGS AND CANDIDATE/COMPARATOR CLASSIFICATION
# =============================================================================
print("[5/N] Computing MOUD exposure and classifying candidates...\n")

def med_match_any(meds, drug_list):
    """Check if any medication in the patient's list matches any drug in drug_list (substring)."""
    return any(any(d in m_name for d in drug_list) for m_name, _ in meds)

def med_match_any_in_window(meds, drug_list, index_date, window_days):
    """Same as above but requires the prescription to be within window_days before index_date."""
    if index_date is None or not HAS_MED_DATE:
        # Fallback to ever-prescribed
        return med_match_any(meds, drug_list)
    cutoff = index_date - timedelta(days=window_days)
    for m_name, m_date in meds:
        if m_date is None:
            continue
        if cutoff <= m_date <= index_date:
            if any(d in m_name for d in drug_list):
                return True
    return False

# MOUD flags (ever-prescribed for candidate classification)
for mrn, pdata in patient_data.items():
    meds = pdata['meds']
    pdata['has_methadone'] = med_match_any(meds, MOUD_METHADONE)
    pdata['has_buprenorphine'] = med_match_any(meds, MOUD_BUPRENORPHINE)
    pdata['has_naltrexone'] = med_match_any(meds, MOUD_NALTREXONE)
    pdata['moud_agent_count'] = sum([
        pdata['has_methadone'],
        pdata['has_buprenorphine'],
        pdata['has_naltrexone']
    ])
    pdata['has_any_moud'] = pdata['moud_agent_count'] >= 1
    pdata['has_2plus_moud'] = pdata['moud_agent_count'] >= 2

    # Qualifier count (candidate funnel definition)
    # ≥2 MOUD agents counts as treatment-refractory proxy (weight 2)
    qualifier_count = (
        int(pdata['has_ptsd']) +
        int(pdata['has_depression']) +
        int(pdata['has_anxiety']) +
        int(pdata['has_tbi']) +
        int(pdata['has_any_moud']) +
        int(pdata['has_2plus_moud'])  # double-counts for treatment-refractory weighting
    )
    pdata['qualifier_count'] = qualifier_count

    if qualifier_count >= 2:
        pdata['cohort'] = 'Candidate'
    elif qualifier_count == 0:
        pdata['cohort'] = 'Comparator'
    else:
        pdata['cohort'] = 'Middle'

# Summary
cohort_counts = Counter(p['cohort'] for p in patient_data.values())
print("Cohort classification:")
for label in ['Candidate', 'Middle', 'Comparator']:
    n_c = cohort_counts.get(label, 0)
    print(f"  {label}: {n_c:,} ({100*n_c/len(patient_data):.1f}%)")

print("\nMOUD agent distribution:")
for k in range(4):
    n_k = sum(1 for p in patient_data.values() if p['moud_agent_count'] == k)
    print(f"  {k} agents: {n_k:,}")
print("="*70)

[5/N] Computing MOUD exposure and classifying candidates...

Cohort classification:
  Candidate: 1,704 (38.2%)
  Middle: 1,447 (32.5%)
  Comparator: 1,308 (29.3%)

MOUD agent distribution:
  0 agents: 2,831
  1 agents: 1,483
  2 agents: 143
  3 agents: 2


In [9]:
# =============================================================================
# CELL 9: CYP2D6 INHIBITOR FLAGS — ACTIVE WINDOW (90-DAY LOOKBACK)
# =============================================================================
# This is the KEY v4 improvement: phenoconversion requires active prescription,
# not 'ever prescribed'. If no dates available, falls back to ever-prescribed
# and flags that for the manuscript limitations section.
# =============================================================================
print(f"[6/N] Computing CYP2D6 inhibitor exposure (window: {ACTIVE_WINDOW_DAYS} days)...\n")

for mrn, pdata in patient_data.items():
    meds = pdata['meds']
    idx_date = pdata['index_date']

    pdata['active_strong_cyp2d6'] = med_match_any_in_window(
        meds, CYP2D6_STRONG, idx_date, ACTIVE_WINDOW_DAYS)
    pdata['active_moderate_cyp2d6'] = med_match_any_in_window(
        meds, CYP2D6_MODERATE, idx_date, ACTIVE_WINDOW_DAYS)
    pdata['active_weak_cyp2d6'] = med_match_any_in_window(
        meds, CYP2D6_WEAK, idx_date, ACTIVE_WINDOW_DAYS)
    pdata['active_strong_or_mod_cyp2d6'] = (
        pdata['active_strong_cyp2d6'] or pdata['active_moderate_cyp2d6']
    )

# Individual drug-level flags for transparency
for drug_name in ['bupropion', 'paroxetine', 'fluoxetine', 'duloxetine', 'sertraline']:
    for mrn, pdata in patient_data.items():
        key = f"active_{drug_name}"
        pdata[key] = med_match_any_in_window(
            pdata['meds'], [drug_name], pdata['index_date'], ACTIVE_WINDOW_DAYS)

# Summary
n = len(patient_data)
print("CYP2D6 inhibitor exposure (overall OUD cohort):")
print(f"  Bupropion:           {count_flag('active_bupropion'):,} ({100*count_flag('active_bupropion')/n:.1f}%)")
print(f"  Paroxetine:          {count_flag('active_paroxetine'):,} ({100*count_flag('active_paroxetine')/n:.1f}%)")
print(f"  Fluoxetine:          {count_flag('active_fluoxetine'):,} ({100*count_flag('active_fluoxetine')/n:.1f}%)")
print(f"  Duloxetine (mod):    {count_flag('active_duloxetine'):,} ({100*count_flag('active_duloxetine')/n:.1f}%)")
print(f"  Sertraline (weak):   {count_flag('active_sertraline'):,} ({100*count_flag('active_sertraline')/n:.1f}%)")
print(f"  ANY strong:          {count_flag('active_strong_cyp2d6'):,} ({100*count_flag('active_strong_cyp2d6')/n:.1f}%)")
print(f"  ANY strong/moderate: {count_flag('active_strong_or_mod_cyp2d6'):,} ({100*count_flag('active_strong_or_mod_cyp2d6')/n:.1f}%)")
print("="*70)

[6/N] Computing CYP2D6 inhibitor exposure (window: 90 days)...

CYP2D6 inhibitor exposure (overall OUD cohort):
  Bupropion:           358 (8.0%)
  Paroxetine:          97 (2.2%)
  Fluoxetine:          190 (4.3%)
  Duloxetine (mod):    381 (8.5%)
  Sertraline (weak):   354 (7.9%)
  ANY strong:          606 (13.6%)
  ANY strong/moderate: 939 (21.1%)


In [10]:
# =============================================================================
# CELL 10: hERG CUMULATIVE LOAD (TIER 1 CATEGORICAL)
# =============================================================================
# Weights: HIGH potency = 3, MODERATE = 1, LOW = 0.3
# Active-window medication exposure, sum across all matched drugs.
# Note on double-counting: some drugs appear in multiple drug lists (e.g.
# paroxetine is both CYP2D6 strong inhibitor and hERG high-potency blocker).
# We compute one hERG score per patient — a drug contributes its hERG weight
# only, not additional weight from CYP2D6 categorization.
# =============================================================================
print("[7/N] Computing cumulative hERG blockade load...\n")

def herg_load(meds, index_date, window_days):
    """Return (load_score, high_count, mod_count, low_count, on_methadone)."""
    # Track which drugs (by key) are present in window to avoid double-counting
    # the same drug if prescribed multiple times
    matched = {'high': set(), 'mod': set(), 'low': set()}
    on_methadone = False

    # Build lookup: med_name_substring -> (tier, weight)
    def check_meds_in_window(meds, index_date, window_days):
        hits = []
        if index_date is None or not HAS_MED_DATE:
            # Ever-prescribed fallback
            for m_name, _ in meds:
                hits.append(m_name)
        else:
            cutoff = index_date - timedelta(days=window_days)
            for m_name, m_date in meds:
                if m_date is None:
                    continue
                if cutoff <= m_date <= index_date:
                    hits.append(m_name)
        return hits

    hits = check_meds_in_window(meds, index_date, window_days)

    for m_name in hits:
        if 'methadone' in m_name:
            on_methadone = True
        # Check high tier first (higher precedence)
        for drug_key, weight in HERG_HIGH.items():
            if drug_key in m_name:
                matched['high'].add(drug_key)
                break
        else:
            # Check moderate if not high
            # Special case: citalopram substring matches escitalopram; require exact-ish
            for drug_key, weight in HERG_MODERATE.items():
                if drug_key == 'citalopram':
                    if 'citalopram' in m_name and 'escitalopram' not in m_name:
                        matched['mod'].add(drug_key)
                        break
                elif drug_key in m_name:
                    matched['mod'].add(drug_key)
                    break
            else:
                for drug_key, weight in HERG_LOW.items():
                    if drug_key in m_name:
                        matched['low'].add(drug_key)
                        break

    high_count = len(matched['high'])
    mod_count = len(matched['mod'])
    low_count = len(matched['low'])
    score = 3.0 * high_count + 1.0 * mod_count + 0.3 * low_count
    return score, high_count, mod_count, low_count, on_methadone

for mrn, pdata in patient_data.items():
    score, h, m, l, on_meth = herg_load(pdata['meds'], pdata['index_date'], ACTIVE_WINDOW_DAYS)
    pdata['herg_load'] = score
    pdata['herg_high_count'] = h
    pdata['herg_mod_count'] = m
    pdata['herg_low_count'] = l
    pdata['herg_any'] = (h + m + l) > 0
    pdata['herg_high_load'] = score >= 3.0  # At least one high-potency drug
    pdata['on_methadone_active'] = on_meth

# Load distribution overall
n = len(patient_data)
print("hERG load distribution (overall OUD cohort):")
tiers = [
    ('No hERG drugs',     lambda s: s == 0),
    ('Load 0.1-1.0',      lambda s: 0 < s <= 1.0),
    ('Load 1.1-3.0',      lambda s: 1.0 < s <= 3.0),
    ('Load 3.1-6.0',      lambda s: 3.0 < s <= 6.0),
    ('Load >6.0',         lambda s: s > 6.0),
]
for label, fn in tiers:
    cnt = sum(1 for p in patient_data.values() if fn(p['herg_load']))
    print(f"  {label:25s}: {cnt:>6,} ({100*cnt/n:5.1f}%)")

mean_load = np.mean([p['herg_load'] for p in patient_data.values()])
median_load = np.median([p['herg_load'] for p in patient_data.values()])
print(f"\n  Mean load:   {mean_load:.2f}")
print(f"  Median load: {median_load:.2f}")
print("="*70)

[7/N] Computing cumulative hERG blockade load...

hERG load distribution (overall OUD cohort):
  No hERG drugs            :  1,888 ( 42.3%)
  Load 0.1-1.0             :    809 ( 18.1%)
  Load 1.1-3.0             :    775 ( 17.4%)
  Load 3.1-6.0             :    747 ( 16.8%)
  Load >6.0                :    240 (  5.4%)

  Mean load:   1.74
  Median load: 1.00


In [11]:
# =============================================================================
# CELL 11: CONSTRUCT MASTER PATIENT-LEVEL DATAFRAME
# =============================================================================
# Convert patient_data dict to DataFrame for the analysis tables
# =============================================================================
print("[8/N] Building master dataframe...\n")

records = []
for mrn, pdata in patient_data.items():
    records.append({
        'patient_id': mrn,
        'age': pdata.get('age'),
        'sex': pdata.get('sex'),
        'race': pdata.get('race'),
        'cohort': pdata['cohort'],
        'qualifier_count': pdata['qualifier_count'],
        'has_ptsd': pdata['has_ptsd'],
        'has_depression': pdata['has_depression'],
        'has_anxiety': pdata['has_anxiety'],
        'has_tbi': pdata['has_tbi'],
        'has_methadone': pdata['has_methadone'],
        'has_buprenorphine': pdata['has_buprenorphine'],
        'has_naltrexone': pdata['has_naltrexone'],
        'moud_agent_count': pdata['moud_agent_count'],
        'has_any_moud': pdata['has_any_moud'],
        'has_long_qt': pdata['has_long_qt'],
        'has_varr': pdata['has_varr'],
        'has_hf': pdata['has_hf'],
        'has_ckd_3_4': pdata['has_ckd_3_4'],
        'has_ckd_5_esrd': pdata['has_ckd_5_esrd'],
        'has_any_cardiac_renal': pdata['has_any_cardiac_renal'],
        'active_bupropion': pdata['active_bupropion'],
        'active_paroxetine': pdata['active_paroxetine'],
        'active_fluoxetine': pdata['active_fluoxetine'],
        'active_duloxetine': pdata['active_duloxetine'],
        'active_sertraline': pdata['active_sertraline'],
        'active_strong_cyp2d6': pdata['active_strong_cyp2d6'],
        'active_moderate_cyp2d6': pdata['active_moderate_cyp2d6'],
        'active_strong_or_mod_cyp2d6': pdata['active_strong_or_mod_cyp2d6'],
        'herg_load': pdata['herg_load'],
        'herg_high_count': pdata['herg_high_count'],
        'herg_mod_count': pdata['herg_mod_count'],
        'herg_low_count': pdata['herg_low_count'],
        'herg_any': pdata['herg_any'],
        'herg_high_load': pdata['herg_high_load'],
        'on_methadone_active': pdata['on_methadone_active'],
    })

df = pd.DataFrame(records)
print(f"✓ Master dataframe: {len(df):,} patients × {len(df.columns)} variables")
print(f"\nCohort breakdown:")
print(df['cohort'].value_counts())
print("="*70)

[8/N] Building master dataframe...

✓ Master dataframe: 4,459 patients × 36 variables

Cohort breakdown:
Candidate     1704
Middle        1447
Comparator    1308
Name: cohort, dtype: int64


In [12]:
# =============================================================================
# CELL 12: TABLE 1 — COHORT DERIVATION
# =============================================================================
print("TABLE 1: COHORT DERIVATION\n")

n_total = len(df)
rows = [
    ('All OUD patients', n_total, 100.0),
    ('OUD + PTSD', int(df['has_ptsd'].sum()), None),
    ('OUD + Depression', int(df['has_depression'].sum()), None),
    ('OUD + Anxiety', int(df['has_anxiety'].sum()), None),
    ('OUD + TBI', int(df['has_tbi'].sum()), None),
    ('OUD + Any MOUD', int(df['has_any_moud'].sum()), None),
    ('OUD + ≥2 MOUD agents', int((df['moud_agent_count']>=2).sum()), None),
    ('Candidates (≥2 qualifiers)', int((df['cohort']=='Candidate').sum()), None),
    ('Middle (1 qualifier)', int((df['cohort']=='Middle').sum()), None),
    ('Comparators (0 qualifiers)', int((df['cohort']=='Comparator').sum()), None),
]
table1 = pd.DataFrame([
    {'Criterion': r[0], 'N': r[1], 'Pct of OUD': r[2] if r[2] is not None else round(100*r[1]/n_total, 1)}
    for r in rows
])
print(table1.to_string(index=False))
table1.to_csv(f'{OUTPUT_DIR}crdw_v4_table1_cohort.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_table1_cohort.csv")
print("="*70)

TABLE 1: COHORT DERIVATION

                  Criterion     N  Pct of OUD
           All OUD patients  4459       100.0
                 OUD + PTSD   356         8.0
           OUD + Depression  1598        35.8
              OUD + Anxiety  1845        41.4
                  OUD + TBI    93         2.1
             OUD + Any MOUD  1628        36.5
       OUD + ≥2 MOUD agents   145         3.3
 Candidates (≥2 qualifiers)  1704        38.2
       Middle (1 qualifier)  1447        32.5
 Comparators (0 qualifiers)  1308        29.3

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_table1_cohort.csv


In [13]:
# =============================================================================
# CELL 13: TABLE 2 — QT/TdP RISK MARKERS (CANDIDATES vs COMPARATORS)
# =============================================================================
print("TABLE 2: QT/TdP RISK MARKERS\n")

cand = df[df['cohort']=='Candidate']
comp = df[df['cohort']=='Comparator']
n_cand = len(cand)
n_comp = len(comp)

def row(name, col):
    cn = int(cand[col].sum())
    cp = int(comp[col].sum())
    return {
        'Marker': name,
        'Cand N': cn,
        'Cand %': round(100*cn/n_cand, 1) if n_cand else None,
        'Comp N': cp,
        'Comp %': round(100*cp/n_comp, 1) if n_comp else None,
        'Fold Δ': round((cn/n_cand)/(cp/n_comp), 2) if (n_comp and cp>0) else None,
    }

markers = [
    ('Long QT syndrome', 'has_long_qt'),
    ('Ventricular arrhythmia', 'has_varr'),
    ('Heart failure', 'has_hf'),
    ('CKD stage 3-4', 'has_ckd_3_4'),
    ('CKD stage 5 / ESRD', 'has_ckd_5_esrd'),
    ('Any cardiac/renal flag', 'has_any_cardiac_renal'),
]

table2 = pd.DataFrame([row(n, c) for n, c in markers])
print(table2.to_string(index=False))
table2.to_csv(f'{OUTPUT_DIR}crdw_v4_table2_qt_risk.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_table2_qt_risk.csv")

# Chi-square test for any cardiac/renal
from scipy.stats import chi2_contingency
a = int(cand['has_any_cardiac_renal'].sum())
b = n_cand - a
c = int(comp['has_any_cardiac_renal'].sum())
d = n_comp - c
chi2, pval, _, _ = chi2_contingency([[a,b],[c,d]])
print(f"\nAny cardiac/renal flag: χ² = {chi2:.2f}, p = {pval:.4g}")
print("="*70)

TABLE 2: QT/TdP RISK MARKERS

 Cand %  Cand N  Comp %  Comp N  Fold Δ                  Marker
    0.8      14     0.5       7    1.54        Long QT syndrome
    5.9     101     4.1      54    1.44  Ventricular arrhythmia
   14.5     247    14.1     184    1.03           Heart failure
    7.3     124     6.6      86    1.11           CKD stage 3-4
    3.0      51     2.9      38    1.03      CKD stage 5 / ESRD
   24.4     415    21.3     279    1.14  Any cardiac/renal flag

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_table2_qt_risk.csv

Any cardiac/renal flag: χ² = 3.65, p = 0.05614


In [14]:
# =============================================================================
# CELL 14: TABLE 3 — CYP2D6 INHIBITOR EXPOSURE (ACTIVE WINDOW)
# =============================================================================
print("TABLE 3: CYP2D6 INHIBITOR EXPOSURE (active-window)\n")

markers = [
    ('Bupropion (strong)', 'active_bupropion'),
    ('Paroxetine (strong)', 'active_paroxetine'),
    ('Fluoxetine (strong)', 'active_fluoxetine'),
    ('ANY strong inhibitor', 'active_strong_cyp2d6'),
    ('Duloxetine (moderate)', 'active_duloxetine'),
    ('Sertraline (weak)', 'active_sertraline'),
    ('Strong or moderate (effective phenoconversion)', 'active_strong_or_mod_cyp2d6'),
]

table3 = pd.DataFrame([row(n, c) for n, c in markers])
print(table3.to_string(index=False))
table3.to_csv(f'{OUTPUT_DIR}crdw_v4_table3_cyp2d6.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_table3_cyp2d6.csv")

# Chi-square for strong inhibitor
a = int(cand['active_strong_cyp2d6'].sum())
b = n_cand - a
c = int(comp['active_strong_cyp2d6'].sum())
d = n_comp - c
chi2, pval, _, _ = chi2_contingency([[a,b],[c,d]])
print(f"\nStrong CYP2D6 inhibitor: χ² = {chi2:.2f}, p = {pval:.4g}")

# Also compute fold-differential explicitly
if n_comp and c>0:
    fold = (a/n_cand) / (c/n_comp)
    print(f"Candidate-vs-comparator fold differential: {fold:.2f}x")
print("="*70)

TABLE 3: CYP2D6 INHIBITOR EXPOSURE (active-window)

 Cand %  Cand N  Comp %  Comp N  Fold Δ                                          Marker
   14.7     250     1.9      25    7.68                              Bupropion (strong)
    3.5      60     0.9      12    3.84                             Paroxetine (strong)
    8.2     140     0.8      10   10.75                             Fluoxetine (strong)
   24.2     413     3.7      48    6.60                            ANY strong inhibitor
   13.4     228     3.5      46    3.80                           Duloxetine (moderate)
   14.4     246     2.9      38    4.97                               Sertraline (weak)
   34.2     582     8.6     112    3.99  Strong or moderate (effective phenoconversion)

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_table3_cyp2d6.csv

Strong CYP2D6 inhibitor: χ² = 239.89, p = 4.147e-54
Candidate-vs-comparator fold differential: 6.60x


In [15]:
# =============================================================================
# CELL 15: TABLE 4 — hERG CUMULATIVE LOAD DISTRIBUTION
# =============================================================================
print("TABLE 4: hERG CUMULATIVE LOAD DISTRIBUTION\n")

def load_tier(s):
    if s == 0: return 'None'
    elif s <= 1.0: return 'Low (0.1-1.0)'
    elif s <= 3.0: return 'Moderate (1.1-3.0)'
    elif s <= 6.0: return 'High (3.1-6.0)'
    else: return 'Very high (>6.0)'

df['herg_tier'] = df['herg_load'].apply(load_tier)

tier_order = ['None', 'Low (0.1-1.0)', 'Moderate (1.1-3.0)', 'High (3.1-6.0)', 'Very high (>6.0)']

table4_rows = []
for tier in tier_order:
    cn = int(((df['cohort']=='Candidate') & (df['herg_tier']==tier)).sum())
    cp = int(((df['cohort']=='Comparator') & (df['herg_tier']==tier)).sum())
    table4_rows.append({
        'hERG Load Tier': tier,
        'Cand N': cn,
        'Cand %': round(100*cn/n_cand, 1) if n_cand else None,
        'Comp N': cp,
        'Comp %': round(100*cp/n_comp, 1) if n_comp else None,
    })

table4 = pd.DataFrame(table4_rows)
print(table4.to_string(index=False))
table4.to_csv(f'{OUTPUT_DIR}crdw_v4_table4_herg_load.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_table4_herg_load.csv")

# Summary statistics
print(f"\nhERG load summary:")
print(f"  Candidates  mean={cand['herg_load'].mean():.2f}  median={cand['herg_load'].median():.2f}")
print(f"  Comparators mean={comp['herg_load'].mean():.2f}  median={comp['herg_load'].median():.2f}")

# Methadone-stratified
cand_meth = cand[cand['on_methadone_active']==True]
cand_nonmeth = cand[cand['on_methadone_active']==False]
print(f"\n  Candidates on methadone    (N={len(cand_meth):,}): mean load={cand_meth['herg_load'].mean():.2f}")
print(f"  Candidates NOT on methadone (N={len(cand_nonmeth):,}): mean load={cand_nonmeth['herg_load'].mean():.2f}")
print("="*70)

TABLE 4: hERG CUMULATIVE LOAD DISTRIBUTION

 Cand %  Cand N  Comp %  Comp N      hERG Load Tier
   20.9     356    69.2     905                None
   17.4     296    15.1     197       Low (0.1-1.0)
   23.3     397     9.0     118  Moderate (1.1-3.0)
   27.1     461     6.2      81      High (3.1-6.0)
   11.4     194     0.5       7    Very high (>6.0)

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_table4_herg_load.csv

hERG load summary:
  Candidates  mean=2.89  median=2.00
  Comparators mean=0.62  median=0.00

  Candidates on methadone    (N=388): mean load=3.92
  Candidates NOT on methadone (N=1,316): mean load=2.58


In [16]:
# =============================================================================
# CELL 16: TABLE 5 — COMPOUND RISK CROSS-TABULATION (cardiac × CYP2D6 × hERG)
# =============================================================================
# The 2×2×2 of compounded vulnerability — identifies patients with multiple
# risk dimensions simultaneously. Key cell: all three present.
# =============================================================================
print("TABLE 5: COMPOUND RISK (Candidates only)\n")

# Build compound flags for candidates
cand_cpr = cand.copy()
cand_cpr['flag_cardiac'] = cand_cpr['has_any_cardiac_renal'].astype(int)
cand_cpr['flag_cyp'] = cand_cpr['active_strong_or_mod_cyp2d6'].astype(int)
cand_cpr['flag_herg'] = cand_cpr['herg_high_load'].astype(int)

crosstab = cand_cpr.groupby(['flag_cardiac','flag_cyp','flag_herg']).size().reset_index(name='n')
crosstab['pct_of_candidates'] = round(100 * crosstab['n'] / n_cand, 1)
crosstab = crosstab.sort_values(['flag_cardiac','flag_cyp','flag_herg'], ascending=[False,False,False])

print(crosstab.to_string(index=False))

# Highlight the all-three cell
all_three_n = int(((cand_cpr['flag_cardiac']==1) & (cand_cpr['flag_cyp']==1) & (cand_cpr['flag_herg']==1)).sum())
all_three_pct = 100*all_three_n/n_cand
print(f"\n>>> HIGHEST-RISK CELL: cardiac + CYP2D6 + high hERG load <<<")
print(f"    N = {all_three_n:,}  ({all_three_pct:.1f}% of candidates)")

crosstab.to_csv(f'{OUTPUT_DIR}crdw_v4_table5_compound_risk.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_table5_compound_risk.csv")
print("="*70)

TABLE 5: COMPOUND RISK (Candidates only)

 flag_cardiac  flag_cyp  flag_herg    n  pct_of_candidates
            1         1          1  100                5.9
            1         1          0   50                2.9
            1         0          1  108                6.3
            1         0          0  157                9.2
            0         1          1  293               17.2
            0         1          0  139                8.2
            0         0          1  270               15.8
            0         0          0  587               34.4

>>> HIGHEST-RISK CELL: cardiac + CYP2D6 + high hERG load <<<
    N = 100  (5.9% of candidates)

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_table5_compound_risk.csv


In [17]:
# =============================================================================
# CELL 17: TABLE 6 — SCREENING BURDEN SCENARIOS
# =============================================================================
print("TABLE 6: SCREENING BURDEN SCENARIOS\n")

# Composite includes cardiac/renal OR strong CYP2D6 OR high hERG load
df['composite_screen'] = (
    df['has_any_cardiac_renal'] |
    df['active_strong_cyp2d6'] |
    df['herg_high_load']
).astype(int)

cand = df[df['cohort']=='Candidate']
comp = df[df['cohort']=='Comparator']

scenarios = [
    ('A: Long QT syndrome', 'has_long_qt'),
    ('B: Ventricular arrhythmia', 'has_varr'),
    ('C: Heart failure', 'has_hf'),
    ('D: CKD stage 3-4', 'has_ckd_3_4'),
    ('E: CKD 5 / ESRD', 'has_ckd_5_esrd'),
    ('F: Strong CYP2D6 inhibitor', 'active_strong_cyp2d6'),
    ('G: High hERG load (≥3)', 'herg_high_load'),
    ('H: Any cardiac/renal (A-E)', 'has_any_cardiac_renal'),
    ('I: Composite (cardiac OR CYP2D6 OR hERG)', 'composite_screen'),
]

rows_t6 = []
for name, col in scenarios:
    cn = int(cand[col].sum())
    cp = int(comp[col].sum())
    rows_t6.append({
        'Scenario': name,
        'Cand N': cn,
        'Cand %': round(100*cn/n_cand, 1) if n_cand else None,
        'Comp N': cp,
        'Comp %': round(100*cp/n_comp, 1) if n_comp else None,
        'Fold Δ': round((cn/n_cand)/(cp/n_comp), 2) if (n_comp and cp>0) else None,
    })

table6 = pd.DataFrame(rows_t6)
print(table6.to_string(index=False))
table6.to_csv(f'{OUTPUT_DIR}crdw_v4_table6_screening.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_table6_screening.csv")

# Chi-square for composite
a = int(cand['composite_screen'].sum())
b = n_cand - a
c = int(comp['composite_screen'].sum())
d = n_comp - c
chi2, pval, _, _ = chi2_contingency([[a,b],[c,d]])
print(f"\nComposite screening: χ² = {chi2:.2f}, p = {pval:.4g}")
print("="*70)

TABLE 6: SCREENING BURDEN SCENARIOS

 Cand %  Cand N  Comp %  Comp N  Fold Δ                                  Scenario
    0.8      14     0.5       7    1.54                       A: Long QT syndrome
    5.9     101     4.1      54    1.44                 B: Ventricular arrhythmia
   14.5     247    14.1     184    1.03                          C: Heart failure
    7.3     124     6.6      86    1.11                          D: CKD stage 3-4
    3.0      51     2.9      38    1.03                           E: CKD 5 / ESRD
   24.2     413     3.7      48    6.60                F: Strong CYP2D6 inhibitor
   45.2     771    10.8     141    4.20                    G: High hERG load (≥3)
   24.4     415    21.3     279    1.14                H: Any cardiac/renal (A-E)
   62.9    1072    30.7     401    2.05  I: Composite (cardiac OR CYP2D6 OR hERG)

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_table6_screening.csv

Composite screening: χ² = 306.77, p = 1.102e-68


In [18]:
# =============================================================================
# CELL 18: SENSITIVITY — METHADONE EXCLUSION
# =============================================================================
# Methadone qualifies patients as candidates AND is a hERG drug.
# Re-run composite burden with methadone users excluded.
# =============================================================================
print("SENSITIVITY: METHADONE EXCLUSION\n")

cand_nometh = df[(df['cohort']=='Candidate') & (~df['on_methadone_active'])]
comp_nometh = df[(df['cohort']=='Comparator') & (~df['on_methadone_active'])]

print(f"Candidates excluding active methadone: {len(cand_nometh):,} (from {n_cand:,})")
print(f"Comparators (methadone can't qualify): {len(comp_nometh):,} (from {n_comp:,})")

rows_sens = []
for name, col in [
    ('Strong CYP2D6 inhibitor', 'active_strong_cyp2d6'),
    ('High hERG load', 'herg_high_load'),
    ('Any cardiac/renal', 'has_any_cardiac_renal'),
    ('Composite', 'composite_screen'),
]:
    cn = int(cand_nometh[col].sum())
    cp = int(comp_nometh[col].sum())
    rows_sens.append({
        'Scenario': name,
        'Cand (no meth) N': cn,
        'Cand %': round(100*cn/max(len(cand_nometh),1), 1),
        'Comp (no meth) N': cp,
        'Comp %': round(100*cp/max(len(comp_nometh),1), 1),
    })

sens = pd.DataFrame(rows_sens)
print(sens.to_string(index=False))
sens.to_csv(f'{OUTPUT_DIR}crdw_v4_sens_methadone_excluded.csv', index=False)
print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_sens_methadone_excluded.csv")
print("="*70)

SENSITIVITY: METHADONE EXCLUSION

Candidates excluding active methadone: 1,316 (from 1,704)
Comparators (methadone can't qualify): 1,308 (from 1,308)
 Cand %  Cand (no meth) N  Comp %  Comp (no meth) N                 Scenario
   25.1               330     3.7                48  Strong CYP2D6 inhibitor
   40.8               537    10.8               141           High hERG load
   22.9               302    21.3               279        Any cardiac/renal
   59.4               782    30.7               401                Composite

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_sens_methadone_excluded.csv


In [19]:
# =============================================================================
# CELL 19: SENSITIVITY - QUALIFIER COUNT GRADIENT (pandas-compat)
# =============================================================================
# Does vulnerability scale with qualifier count? Expected: monotonic increase.
# Rewritten to avoid named-aggregation syntax (requires pandas>=0.25) so this
# cell runs on older AMAREL pandas environments.
# =============================================================================
print("SENSITIVITY: QUALIFIER COUNT GRADIENT\n")

g = df.groupby('qualifier_count')
grad = pd.DataFrame({
    'qualifier_count': g['patient_id'].count().index,
    'n':               g['patient_id'].count().values,
    'pct_cardiac':     g['has_any_cardiac_renal'].mean().values,
    'pct_strong_cyp':  g['active_strong_cyp2d6'].mean().values,
    'pct_herg_high':   g['herg_high_load'].mean().values,
    'pct_composite':   g['composite_screen'].mean().values,
})

for col in ['pct_cardiac', 'pct_strong_cyp', 'pct_herg_high', 'pct_composite']:
    grad[col] = (100 * grad[col]).round(1)

print(grad.to_string(index=False))
grad.to_csv(f'{OUTPUT_DIR}crdw_v4_sens_qualifier_gradient.csv', index=False)
print(f"\n\u2713 Saved {OUTPUT_DIR}crdw_v4_sens_qualifier_gradient.csv")
print("=" * 70)


SENSITIVITY: QUALIFIER COUNT GRADIENT

 qualifier_count     n  pct_cardiac  pct_strong_cyp  pct_herg_high  pct_composite
               0  1308         21.3             3.7           10.8           30.7
               1  1447         25.6            10.0           22.0           44.9
               2  1056         24.1            19.9           35.3           55.4
               3   504         26.0            28.2           56.7           72.0
               4   126         22.2            44.4           77.0           86.5
               5    18          5.6            27.8           83.3           83.3

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_sens_qualifier_gradient.csv


In [20]:
# =============================================================================
# CELL 20: SENSITIVITY — AGE 25-55 (MEXICO CLINIC DEMOGRAPHIC)
# =============================================================================
# Published ibogaine clinic demographics are predominantly 25-55, not
# the CRDW average of ~60. Subset analysis checks robustness.
# =============================================================================
print("SENSITIVITY: AGE 25-55 SUBGROUP\n")

df['age_25_55'] = (df['age'].astype(float) >= 25) & (df['age'].astype(float) <= 55)

cand_25_55 = df[(df['cohort']=='Candidate') & df['age_25_55']]
print(f"Candidates age 25-55: {len(cand_25_55):,} ({100*len(cand_25_55)/n_cand:.1f}% of candidates)")

if len(cand_25_55) > 0:
    comp_25_55 = df[(df['cohort']=='Comparator') & df['age_25_55']]
    rows_age = []
    for name, col in [
        ('Strong CYP2D6 inhibitor', 'active_strong_cyp2d6'),
        ('High hERG load', 'herg_high_load'),
        ('Any cardiac/renal', 'has_any_cardiac_renal'),
        ('Composite', 'composite_screen'),
    ]:
        rows_age.append({
            'Scenario': name,
            'Cand 25-55 %': round(100*cand_25_55[col].mean(), 1),
            'Cand all ages %': round(100*cand[col].mean(), 1),
            'Comp 25-55 %': round(100*comp_25_55[col].mean(), 1) if len(comp_25_55) else None,
        })
    age_sens = pd.DataFrame(rows_age)
    print(age_sens.to_string(index=False))
    age_sens.to_csv(f'{OUTPUT_DIR}crdw_v4_sens_age_25_55.csv', index=False)
    print(f"\n✓ Saved {OUTPUT_DIR}crdw_v4_sens_age_25_55.csv")
print("="*70)

SENSITIVITY: AGE 25-55 SUBGROUP

Candidates age 25-55: 649 (38.1% of candidates)
 Cand 25-55 %  Cand all ages %  Comp 25-55 %                 Scenario
         23.7             24.2           2.3  Strong CYP2D6 inhibitor
         47.9             45.2          12.0           High hERG load
         14.2             24.4          10.3        Any cardiac/renal
         60.1             62.9          22.0                Composite

✓ Saved /home/jg1984/paper2_crdw_v4_1/crdw_v4_sens_age_25_55.csv


In [21]:
# =============================================================================
# CELL 21: SAVE PATIENT-LEVEL DATAFRAME AND SUMMARY
# =============================================================================
print("Saving outputs...\n")

# Patient-level (full)
master_path = f'{OUTPUT_DIR}crdw_v4_patient_level.csv'
df.to_csv(master_path, index=False)
print(f"✓ Patient-level: {master_path}  ({len(df):,} rows × {len(df.columns)} cols)")

# Compact summary for manuscript
summary = {
    'run_date': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'study_window_start': STUDY_START,
    'study_window_end': STUDY_END,
    'active_window_days': ACTIVE_WINDOW_DAYS,
    'med_dates_available': HAS_MED_DATE,
    'dx_dates_available': HAS_DX_DATE,
    'n_oud': len(df),
    'n_candidates': int((df['cohort']=='Candidate').sum()),
    'n_comparators': int((df['cohort']=='Comparator').sum()),
    'n_middle': int((df['cohort']=='Middle').sum()),
    'cand_pct_strong_cyp2d6': round(100*cand['active_strong_cyp2d6'].mean(), 1),
    'comp_pct_strong_cyp2d6': round(100*comp['active_strong_cyp2d6'].mean(), 1),
    'cand_pct_any_cardiac_renal': round(100*cand['has_any_cardiac_renal'].mean(), 1),
    'comp_pct_any_cardiac_renal': round(100*comp['has_any_cardiac_renal'].mean(), 1),
    'cand_pct_high_herg': round(100*cand['herg_high_load'].mean(), 1),
    'comp_pct_high_herg': round(100*comp['herg_high_load'].mean(), 1),
    'cand_pct_composite': round(100*cand['composite_screen'].mean(), 1),
    'comp_pct_composite': round(100*comp['composite_screen'].mean(), 1),
    'cand_mean_herg_load': round(cand['herg_load'].mean(), 2),
    'comp_mean_herg_load': round(comp['herg_load'].mean(), 2),
}
summary_df = pd.DataFrame([summary])
summary_path = f'{OUTPUT_DIR}crdw_v4_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary:       {summary_path}")

print(f"\nAll v4 analysis outputs in: {OUTPUT_DIR}")
print("Files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(OUTPUT_DIR + f) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

print(f"\n✓ Complete: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("="*70)

Saving outputs...

✓ Patient-level: /home/jg1984/paper2_crdw_v4_1/crdw_v4_patient_level.csv  (4,459 rows × 39 cols)
✓ Summary:       /home/jg1984/paper2_crdw_v4_1/crdw_v4_summary.csv

All v4 analysis outputs in: /home/jg1984/paper2_crdw_v4_1/
Files:
  crdw_v4_patient_level.csv  (1073.8 KB)
  crdw_v4_sens_age_25_55.csv  (0.2 KB)
  crdw_v4_sens_methadone_excluded.csv  (0.2 KB)
  crdw_v4_sens_qualifier_gradient.csv  (0.2 KB)
  crdw_v4_summary.csv  (0.5 KB)
  crdw_v4_table1_cohort.csv  (0.3 KB)
  crdw_v4_table2_qt_risk.csv  (0.3 KB)
  crdw_v4_table3_cyp2d6.csv  (0.3 KB)
  crdw_v4_table4_herg_load.csv  (0.2 KB)
  crdw_v4_table5_compound_risk.csv  (0.2 KB)
  crdw_v4_table6_screening.csv  (0.4 KB)

✓ Complete: 2026-04-21 23:28


In [22]:
# =============================================================================
# CELL 22: QA / SANITY CHECKS
# =============================================================================
# Run these and confirm before using numbers in the manuscript.
# =============================================================================
print("QA CHECKS\n")

print(f"1. Total OUD reconciliation: {len(df):,}")
print(f"   Expected from previous runs: ~4,459 (prior analysis)")
print()
print(f"2. Cohort sum: {(df['cohort']=='Candidate').sum() + (df['cohort']=='Middle').sum() + (df['cohort']=='Comparator').sum():,} = {len(df):,} ✓" if (df['cohort'].isin(['Candidate','Middle','Comparator'])).all() else "FAIL")
print()
print(f"3. Age sanity: min={df['age'].min()}, max={df['age'].max()}, mean={df['age'].astype(float).mean():.1f}")
print()
print(f"4. hERG load sanity (candidates):")
print(f"   min={cand['herg_load'].min():.2f}, max={cand['herg_load'].max():.2f}, mean={cand['herg_load'].mean():.2f}")
print()
print(f"5. Escitalopram/citalopram sanity check:")
# Count patients flagged with active citalopram (should exclude escitalopram)
escit_only = sum(1 for p in patient_data.values()
                 if any('escitalopram' in m and (p['index_date'] is None or (m_d is not None and p['index_date'] and (p['index_date'] - m_d).days <= ACTIVE_WINDOW_DAYS))
                        for m, m_d in p['meds']))
print(f"   Patients with any escitalopram prescription (sanity): {escit_only:,}")
print()
print(f"6. Differential preservation check:")
ratio = (cand['active_strong_cyp2d6'].mean()) / max(comp['active_strong_cyp2d6'].mean(), 0.001)
print(f"   Strong CYP2D6 candidate/comparator ratio: {ratio:.2f}x (All of Us v5 ratio = 7.1x)")
print(f"   If CRDW active-window gives similar ratio → confirms differential is robust")
print(f"   If ratio differs substantially → flag for investigation")
print()

# Flag if things look off
flags = []
if len(df) < 3000 or len(df) > 10000:
    flags.append(f"OUD count unusual: {len(df)}")
if cand['herg_load'].max() > 20:
    flags.append(f"hERG max very high: {cand['herg_load'].max()}")
if not HAS_MED_DATE:
    flags.append("No medication dates — active-window disabled (limitation for manuscript)")

if flags:
    print("⚠️  INVESTIGATION FLAGS:")
    for f in flags:
        print(f"   - {f}")
else:
    print("✓ No QA flags raised")

print("="*70)

QA CHECKS

1. Total OUD reconciliation: 4,459
   Expected from previous runs: ~4,459 (prior analysis)

2. Cohort sum: 4,459 = 4,459 ✓

3. Age sanity: min=13, max=90, mean=60.1

4. hERG load sanity (candidates):
   min=0.00, max=18.30, mean=2.89

5. Escitalopram/citalopram sanity check:
   Patients with any escitalopram prescription (sanity): 367

6. Differential preservation check:
   Strong CYP2D6 candidate/comparator ratio: 6.60x (All of Us v5 ratio = 7.1x)
   If CRDW active-window gives similar ratio → confirms differential is robust
   If ratio differs substantially → flag for investigation

⚠️  INVESTIGATION FLAGS:
   - No medication dates — active-window disabled (limitation for manuscript)
